# 01 · CaRS-50 — build the pool

*Swales CARS moves in research-article introductions (3 or 11 classes)*

### Where this sits

```
▶ 01 build the pool  →  02 sample  →  03 annotate  →  04 prompt  →  05 report
```

You run **01 once per group**, for your own track only. It ends by writing `data/pools/<track>_pool.json` — the file notebook 02 opens.

---

**What it is.** 50 BioRxiv article introductions, annotated sentence by sentence with Swales' CARS Move and Step scheme. **The annotators themselves reached only κ ≈ 0.43** — so on this track, "the model is wrong" and "the scheme is fuzzy" are both live explanations, and telling them apart is the interesting part.

**Difficulty of the labeling judgment:** ★★★ — hard. Judging moves in an introduction needs more context than a single sentence gives you.

**Licence:** CC BY 4.0  
**Cite:** Lam, C. & Nnamoko, N. (2025). *Mendeley Data*, V1. doi:10.17632/kwr9s5c4nk.1

---

Every dataset in this course is reshaped into the **same canonical schema**, so one pipeline works for all of them:

```json
[{"id": 1, "text": "...", "label": "..."}]
```

The *raw* data, though, looks different every time. **That difference is the lesson** — half of building a gold standard is getting messy real data into a clean, consistent shape.

Those three keys are required on every track. Two tracks add more: `cars50` and `raamove` ask what a sentence *does in a passage*, which is not always decidable from the sentence on its own, so their items also carry `doc_id`, `sent_index`, `n_sents` and `context`. Extra keys are safe everywhere — nothing in the pipeline checks for keys it does not need.

> The reshaping code below is read straight out of `scripts/reshape.py` — it is the same code `scripts/prep_datasets.py` runs, not a copy of it. What is *missing* from it is missing on purpose: the ✏️ cells are the decisions, and they are yours. (Generated by `scripts/_generate_pool_notebooks.py`; edit that or `reshape.py`, never the `.ipynb`.)

## Setup — run this first

This cell mounts your Google Drive and finds your group's shared folder, `lda2-final-template`. Everything the project produces — the pool, the gold set, your prompts, the outputs — is an ordinary file in there, which is what makes it survive the runtime resetting *and* lets the rest of your group see it.

**One member sets the folder up once:**

1. That member runs the `git clone` line this cell prints if the folder is missing, which puts it in their own Drive.
2. They share it with the group (right-click ▸ *Share*), with edit access.
3. Everyone else opens *Shared with me*, right-clicks the folder, and chooses **Add shortcut to Drive** ▸ *My Drive*.

Keep that shortcut's name exactly `lda2-final-template`. It is what makes the same path work for all of you — if Drive renames it to `lda2-final-template (1)`, this cell will not find it.

From then on, open notebooks from the folder itself (*File ▸ Open notebook ▸ Drive*) rather than from the GitHub badge, so you are working on your group's copy and not a fresh one.

In [ ]:
# ------------------------------------------------------------------
# SETUP — run me first. You are not expected to read it.
# ------------------------------------------------------------------
# This cell is plumbing, and it is the only cell in the project that is.
# It finds your group's shared folder in Google Drive, because everything
# this project keeps goes in there: a Colab runtime is wiped when it resets,
# and nobody else in your group can see inside it. Then it makes the
# project's own code importable. Run it and move on; nothing below asks you
# to have understood it.

FOLDER = "lda2-final-template"     # the shared folder, in every member's Drive

import os, sys

PROJECT = ".."                              # running locally: it is just above us

try:
    from google.colab import drive           # only exists inside Colab
except ImportError:
    pass
else:
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/" + FOLDER
    if not os.path.isdir(PROJECT):
        raise RuntimeError(
            "Could not find " + PROJECT + "\n\n"
            "Setting the folder up for your group? Run this in a new cell:\n"
            "  !git clone https://github.com/egumasa/lda2-final-template.git "
            + PROJECT + "\n"
            "then share the folder with the rest of your group.\n\n"
            "Someone else already did? Open Drive, find the folder under "
            "'Shared with me', right-click it, and choose 'Add shortcut to "
            "Drive'. Keep the name exactly " + FOLDER + ".")
    # Work in the RUNTIME, not in Drive: the next cells download a whole
    # corpus, and raw data is big, mostly not ours to redistribute, and one
    # command to fetch again. The pool you build from it is what persists.
    os.makedirs("/content/raw", exist_ok=True)
    os.chdir("/content/raw")

# scripts/ and config.py, by their real paths - so they are found from wherever
# this notebook happens to be working.
sys.path.append(PROJECT)
sys.path.append(PROJECT + "/scripts")

# Re-read config.yaml every time this cell runs. Without the reload, Python
# hands back the settings it read the FIRST time, and editing config.yaml
# would appear to do nothing until you restarted the runtime.
import importlib
import config
importlib.reload(config)

# Named one by one rather than with `import *`, so that every name a cell
# below uses can be traced back to the file it came from — config.yaml for
# these, scripts/ for the rest.
from config import (TRACK, GROUP, RUN, SEED, N_PER_CLASS, DEV, CODERS,
                    MEMBERS, LABELS_ORDER, TEMPERATURE, MODEL, ROOT, OUT_DIR,
                    POOL_PATH, DEMO_POOL_PATH, SAMPLE_PATH, GOLD_PATH,
                    SAMPLE_BEFORE_TOPUP_PATH, DEV_PATH, TEST_PATH,
                    DISAGREED_PATH, PRED_PATH, ROUNDS_PATH, NOTES_PATH,
                    TESTLOG_PATH, PROMPT_FILE, SHEET_PATH, TRIAGE_PATH,
                    CODER_TRIAGE_PATH, describe)
from pathlib import Path

describe()                  # what this notebook is working on


## Step 1 — Download the raw data

This one is on **Mendeley Data**, which has a public API. We ask it for the dataset's file list, then download each file. The CDN refuses requests that do not look like a browser, hence the `User-Agent` header.

In [ ]:
import json, urllib.request, pathlib

RAW_DIR = pathlib.Path("cars50")     # the folder to download into
RAW_DIR.mkdir(exist_ok=True)

Now we write a small function of our own. `fetch` opens one web address and hands back what is there — with a `User-Agent` header, because the server refuses requests that do not look like they came from a browser.

A `def` gives a name to a few lines so they can be used more than once. This cell prints nothing; the next one calls it.

In [ ]:
def fetch(url):
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    return urllib.request.urlopen(request, timeout=60)

Now we ask Mendeley what files this dataset has, then download each one.

In [ ]:
# One call to the Mendeley API, unpacked a step at a time.
catalogue_url = "https://data.mendeley.com/public-api/datasets/kwr9s5c4nk"
raw_text = fetch(catalogue_url).read()
catalogue = json.loads(raw_text)

for record in catalogue["files"]:
    target = RAW_DIR / record["filename"]
    if not target.exists():
        download_url = record["content_details"]["download_url"]
        target.write_bytes(fetch(download_url).read())

xml_files = sorted(RAW_DIR.glob("*.xml"))
print("downloaded", len(xml_files), "XML files")

## Step 2 — Look at the raw format

**XML** this time. Each sentence carries a `step` code like `1b`:

```xml
<sentence><sentenceID/><text/><step>1b</step></sentence>
```

Now we print the beginning of the first file, so you can see the real thing rather than that sketch of it.

In [ ]:
first_file = xml_files[0]
raw_xml = first_file.read_text(encoding="utf-8")
print(raw_xml[:900])

### Reading XML with `ElementTree`

XML is nested, so unlike a TSV or a CSV you cannot get at a field by position. Python's built-in `xml.etree.ElementTree` parses the file into a tree of elements you then navigate by tag name.

**The nesting, outermost first:**

* `<biology_intro>` — the root element, one per file
* `<fulltext>` — the introduction itself
* `<paragraph>` — one or more per introduction
* `<sentence>` — one or more per paragraph, each carrying:
  * `<sentenceID>` — an identifier
  * `<text>` — the sentence
  * `<step>` — the rhetorical step code, e.g. `1b`

The reshaping function below uses exactly these:

1. `ET.parse(path)` — read one file into a tree.
2. `tree.iter("sentence")` — every `<sentence>` at **any** depth, so you never have to walk the paragraphs yourself. It yields them in document order, which is what lets each sentence keep its place in the introduction.
3. `element.find("text")` — the first child with that tag, or `None` if it is missing. That `None` is why the reshaping code checks before using it.
4. `element.text` — the string inside a tag. It is `None` for an empty tag, hence the `(… or "")` guard before `.strip()`.
5. `path.stem` — the filename without `.xml`, e.g. `text001`. That is the document id. **Not** `<sentenceID>`: those are padded three different ways, mix two widths inside `text038.xml`, and `t025s020` appears twice in `text025.xml`. Position comes from counting, not from reading an id.

Now we parse that same file into a tree, so we can ask it for tags by name instead of hunting through the text.

In [ ]:
import xml.etree.ElementTree as ET

tree = ET.parse(first_file)
print("reading", first_file.name)

Now we walk the first three sentences and print what each one carries.

In [ ]:
# Walk the first three sentences and print their three child tags.
for position, sentence in enumerate(tree.iter("sentence")):
    if position >= 3:                # just the first three, to keep this short
        break
    for tag in ("sentenceID", "text", "step"):
        child = sentence.find(tag)
        # A tag can be missing (child is None) or present but empty
        # (child.text is None). Both mean there is nothing to read.
        if child is None or child.text is None:
            value = "MISSING"
        else:
            value = child.text.strip()
        print(" ", tag + ":", value)
    print()

## Step 3 — Reshape into the canonical schema

The parsing is written for you, and it gives you **both granularities at once**:

- the leading digit of `1b` is the **Move** → 3 classes;
- the whole code `1b` is the **Step** → 11 classes.

Sentences with no code, or a code that does not start with a move digit, are dropped either way. In *this* corpus that guard never actually fires — all 1297 sentences are coded — so do not write "we dropped N malformed sentences" in your report without checking the number first. The guard is there because the next corpus you meet will need it.

✏️ **Which one you study is the decision**, and on this track it is the whole shape of the project. Three classes with a few hundred items each is a fair task you can sample 40 items from comfortably. Eleven classes over the same sentences means some steps have barely a dozen examples, a confusion matrix with 121 cells, and an annotation job your two coders will find genuinely hard — remember the original annotators managed only κ ≈ 0.43 at this granularity.

Neither is the safe answer. The 11-class version makes a better project **if** you have the time to annotate it properly and the nerve to report a low F1 with a good explanation. Decide now, write it in `PLAN.md`, and do not switch after you have seen the numbers.

### Each sentence keeps its introduction

The difficulty note at the top of this notebook says judging a move needs more context than a single sentence gives you. So the code below does not throw the introduction away. It reads each file **twice** — once to collect the whole introduction in order, once to emit the items — and every item comes out carrying four extra fields on top of the canonical three:

| field | what it is |
|---|---|
| `doc_id` | which introduction, e.g. `text001` |
| `sent_index` | where in it, counting from 0 |
| `n_sents` | how many sentences the introduction has |
| `context` | the introduction itself, one sentence per line |

One detail worth arguing about: `context` keeps **every** sentence that has text, including the ones dropped for having no usable code. They belong there because a reader saw them — filtering them out would hand the model a doctored introduction that never existed. Both granularities get identical fields; it is the same sentence in the same passage, just labelled two ways.

Those fields travel with the item all the way: notebook 03 shows the introduction to your two coders, and notebook 04 can put it in the prompt. `prompts/cars50.txt` shows the model the sentence alone, `prompts/cars50_context.txt` shows it the introduction first. These are 26 sentences on average and up to 47, so the context condition is noticeably slower to run — worth knowing before you start it at 4pm.

### The code that does it — read it, then run it

Three functions. `reshape_cars50` is the one to read: it is where the one parse becomes two datasets.

It is read straight out of `scripts/reshape.py` when this notebook is generated, so it is not a simplified copy: it is the code that runs.

It arrives one function per cell, so you can take them one at a time. **None of these cells print anything.** They only give the functions their names — that is what `def` does. You will see no output until the cell *after* them, which calls one.

First, the two library modules the code below needs. `import` is how Python is told to load one.

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path

`reid` renumbers items 1, 2, 3 … so that every item has an id of its own.

In [ ]:
def reid(items: list[dict[str, str]]) -> list[dict[str, str]]:
    """Renumber ids sequentially from 1, keeping the current order.

    Args:
        items: the items to renumber. They are copied, not changed in place.

    Returns:
        The same items with new ids.
    """
    renumbered = []
    next_id = 1

    for item in items:
        ### Copy before writing ###
        new_item = dict(item)                    # Work on a copy, so the caller's item is left alone.

        ### Stamp the id ###
        new_item["id"] = next_id                 # Overwrite whatever id was there with the running number.
        renumbered.append(new_item)              # Keep it in the order it arrived.
        next_id = next_id + 1                    # Advance, so the next item gets a fresh id.

    return renumbered

`_tag_text` reads the text inside one XML tag. It is separate because a tag can be missing *or* present-but-empty, and both have to come back as "". The leading underscore is a convention meaning "a helper for the function below" — it is not a typo, and nothing stops you calling it.

In [ ]:
def _tag_text(element) -> str:
    """The text inside an XML tag, tidied - or "" when the tag is missing or empty.

    Args:
        element: the ElementTree element, which may be None.

    Returns:
        The text, with blank space trimmed. ElementTree gives None both for a tag
        that is not there and for one that is there but empty; both mean "nothing to
        read", so both come back as "".
    """
    if element is None:
        return ""
    if element.text is None:
        return ""
    return element.text.strip()

`reshape_cars50` is the work: one walk through the XML, two lists out — the 3-class moves and the 11-class steps. Read the two passes.

In [ ]:
def reshape_cars50(cars50_dir: str | Path) -> tuple:
    """Parse the 50 XML introductions into TWO datasets: moves, and move+step.

    XML shape:
        <sentence><sentenceID/><text/><step>1b</step></sentence>

    The `step` code is like "1b": the leading DIGIT is the Move, the whole code is the
    Step. So one parse gives two granularities, and which you use is a scheme decision:
    3 classes is a fair task, 11 classes is the stretch version.

    Args:
        cars50_dir: the folder holding the 50 downloaded XML files.

    Returns:
        Two lists: the move items and the move+step items. A move is a rhetorical
        function WITHIN an introduction, so each item also carries the introduction
        it came from.
    """
    source_dir = Path(cars50_dir)
    move_rows = []
    step_rows = []

    ### Walk the 50 XML files ###
    for xml_path in sorted(source_dir.glob("*.xml")):   # One file per article introduction.

        ### Parse one file into a tree ###
        tree = ET.parse(xml_path)                # ElementTree turns the XML into a navigable tree.
        doc_id = xml_path.stem                   # e.g. "text001". The FILENAME - the <sentenceID> tags are not reliable.

        ### PASS 1: read the whole introduction, in order ###
        # Every sentence that has text goes in here, including ones we are about to drop
        # for having no usable code. They belong in the passage because a reader saw them:
        # leaving them out would hand the model a doctored introduction.
        passage = []
        for sentence in tree.iter("sentence"):   # .iter() finds them at any depth, so the paragraph nesting does not matter.
            text_element = sentence.find("text")     # The <text> child, or None if absent.
            step_element = sentence.find("step")     # The <step> child, or None if absent.
            text = _tag_text(text_element)
            if text:
                passage.append((text, step_element))

        texts = []                               # Just the sentences.
        for text, step_element in passage:
            texts.append(text)
        context = "\n".join(texts)               # One string, newlines kept so the sentences stay visible.

        ### PASS 2: emit an item for each sentence that carries a usable code ###
        # Position comes from enumerate(), never from <sentenceID>: those ids are padded
        # three different ways, mix two widths inside text038.xml, and t025s020 appears
        # twice in text025.xml.
        for position, (text, step_element) in enumerate(passage):
            code = _tag_text(step_element)       # e.g. "1b".

            # Skip anything unlabelled, or whose code does not start with a move digit.
            if not code or not code[0].isdigit():
                continue

            ### Where this sentence sits - the same for both granularities ###
            where = {"doc_id": doc_id,           # Which introduction this sentence is from.
                     "sent_index": position,     # Where in it - 0 is the first sentence.
                     "n_sents": len(texts),      # How long the introduction is.
                     "context": context}         # The introduction itself.

            ### Record the SAME sentence at both granularities ###
            move_row = {"id": 0, "text": text, "label": "Move " + code[0]}   # Leading digit only -> 3 classes.
            step_row = {"id": 0, "text": text, "label": code}                # Whole code -> 11 classes.
            move_row.update(where)               # Add doc_id, sent_index, n_sents, context.
            step_row.update(where)
            move_rows.append(move_row)
            step_rows.append(step_row)

    return reid(move_rows), reid(step_rows)      # Two datasets, each with ids running 1..N.

`validate` checks that every item has an id, a text and a label. Nothing calls it here — step 5 does, just before saving.

In [ ]:
def validate(items: list[dict[str, str]],
             allowed: list[str] | None = None) -> None:
    """Check the canonical schema, and raise on the first problem found.

    Deliberately explicit rather than `assert`: assertions vanish under `python -O`,
    and a silently unvalidated dataset is exactly the kind of thing that surfaces as a
    baffling metric three days later.

    Args:
        items: the items to check, each needing "id", "text" and "label".
        allowed: the labels the scheme allows. Left out, any label passes.

    Returns:
        Nothing.

    Raises:
        ValueError: on the first item that is missing a field, has a repeated id, or
            carries a label outside `allowed`.
    """
    seen_ids = set()
    for position, item in enumerate(items):
        where = "Item number " + str(position + 1) + " of " + str(len(items))
        for field in ("id", "text", "label"):
            if field not in item:
                raise ValueError(
                    where + " has no `" + field + "`, and every item needs all three of "
                    "id, text and label.\n"
                    "That item was built by the reshaping step above, so go back to the "
                    "cell where you filled in your own decision and check it puts a "
                    "`" + field + "` on every row.")
        if item["id"] in seen_ids:
            raise ValueError(
                "Two items have the same id (" + str(item["id"]) + "), so one would "
                "overwrite the other in your annotation sheet.\n"
                "reid() renumbers everything 1, 2, 3 - make sure the last line of your "
                "reshaping step hands its rows to it.")
        seen_ids.add(item["id"])
        if not isinstance(item["text"], str) or not item["text"].strip():
            raise ValueError(
                "The item with id " + str(item["id"]) + " has no text - there is nothing "
                "there for a coder or the model to read.\n"
                "Blank rows usually come from the raw file. Skip them in the reshaping "
                "step rather than annotate them.")
        if not isinstance(item["label"], str) or not item["label"]:
            raise ValueError(
                "The item with id " + str(item["id"]) + " has no label.\n"
                "Every item in a pool needs the published label it came with. If your "
                "label mapping does not cover some code in the raw data, either add it "
                "or drop those rows in the reshaping step - do not leave the label blank.")
        if allowed is not None and item["label"] not in allowed:
            raise ValueError(
                "The item with id " + str(item["id"]) + " has the label '"
                + str(item["label"]) + "', which is not one of the labels you allowed:\n"
                "  " + ", ".join(sorted(allowed)) + "\n"
                "Either add it to your label set, or map it onto one of these in the "
                "cell where you wrote your label mapping.")

## Step 3a — Run it

Now we run the parsing over all 50 files. It hands back **two** lists at once — that is what the comma on the left of the `=` means — one labelled at each granularity.

In [ ]:
move_rows, step_rows = reshape_cars50(RAW_DIR)
print("moves:", len(move_rows), " steps:", len(step_rows))

Now we count the classes in each, because how many items the smallest class has is the ceiling on any balanced draw you make in notebook 02.

In [ ]:
def count_labels(items):
    counts = {}
    for item in items:
        label = item["label"]
        if label not in counts:
            counts[label] = 0
        counts[label] = counts[label] + 1
    return counts

print("move classes:", count_labels(move_rows))
print("step classes:", count_labels(step_rows))

## Step 3b — Choose your granularity

Now we pick which of the two schemes your group will actually study — the 3 moves or the 11 steps — and give it the name `rows` that the rest of the notebook uses.

It is one word to change. Spend the time on the argument instead: `PLAN.md` asks you to justify the choice in a sentence.

**Whichever you pick**, the label names in your prompt and on your annotation sheet have to match these exactly — `Move 1`, or `1b`.

In [ ]:
# ✏️ Step 3b · Choose your granularity ───────────────────────────
# Names one of the two schemes as the one you will study, and prints how many
# items it has.
# Creates: rows (a list) — either move_rows or step_rows

# ✏️ 3 moves or 11 steps? One word, and PLAN.md asks you to defend it.

# Change this one word to "step" for the 11-class version.
# It is written as a choice rather than two lines you delete one of,
# because with two live lines the SECOND one silently wins — and you
# would not find out until notebook 03, halfway through annotating.
GRANULARITY = "move"

if GRANULARITY == "move":
    rows = move_rows          # 3 classes: Move 1 · Move 2 · Move 3
elif GRANULARITY == "step":
    rows = step_rows          # 11 classes: 1a · 1b · 2a … the finer scheme
else:
    raise ValueError(
        "GRANULARITY has to be either \"move\" or \"step\", and it says "
        + repr(GRANULARITY) + ". Fix the line above and run this cell again.")

print("studying the", GRANULARITY, "scheme:", len(rows), "items")


## Step 4 — Check the label balance

Now we look at what you chose, in the shape it will actually be annotated in.

Now we count what we have got: how many items, how many of each label, and which fields every item carries.

In [ ]:
# Count how many items carry each label, one item at a time.
label_counts = {}
for item in rows:
    label = item["label"]
    if label not in label_counts:
        label_counts[label] = 0
    label_counts[label] = label_counts[label] + 1

print("total items:", len(rows))
print("label counts:", label_counts)
print("fields per item:", list(rows[0].keys()))

Now we look at three whole items, to see the shape of one.

Where a track carries a `context` (the whole passage a sentence came from), it is shortened here so it does not bury everything else. That only changes what is **printed** — `rows` itself is untouched.

In [ ]:
for item in rows[:3]:
    preview = dict(item)          # a copy, so trimming it changes nothing
    if preview.get("context"):
        preview["context"] = preview["context"][:70] + " …"
    print(preview)
    print("---")

## Step 5 — Save it

Three short cells: check that `config.yaml` agrees which track this is, check the shape of every item, then write the file.

**First, a safety check.** `POOL_PATH` is built from the `track:` line in `config.yaml`. If that still says another track, saving now would write cars50 data into a file belonging to something else — and everything downstream would run perfectly on the wrong data. The first sign of trouble would be labels that make no sense in notebook 03, by which point two people have annotated forty items.

If this cell stops you: open `config.yaml`, set `track:` to `cars50`, save it, then re-run the SETUP cell at the top of this notebook.

In [ ]:
if TRACK not in ['cars50', 'cars50_step']:
    raise RuntimeError(
        "config.yaml says  track: " + str(TRACK) + "  but this is the cars50 "
        "notebook, so saving now would put cars50 data into "
        + POOL_PATH.name + ", which belongs to another track.\n"
        "Open config.yaml, set  track: to one of cars50 · cars50_step,"
        " save it, then re-run the SETUP cell at the top of this notebook.")

print("config.yaml agrees: this is the", TRACK, "track.")

**Now we check the shape of every item.** Everything downstream — the sampling, the annotation sheet, the scoring — assumes each item has an `id`, a `text` and a `label`. A pool that breaks that assumption does not fail here; it fails in notebook 03, after two people have annotated forty items.

`validate` says nothing when all is well. Silence is the pass.

In [ ]:
validate(rows)
print("All", len(rows), "items have an id, a text and a label.")

**Now we write the pool** into your group's Drive folder, under the exact name notebook 02 will look for. Both notebooks get that name from `config.yaml`, so there is nothing to copy or paste between them.

In [ ]:
import json

POOL_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(POOL_PATH, "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
print("Saved", len(rows), "items to", POOL_PATH)

# If you chose steps, set  track: cars50_step  in config.yaml before running this: POOL_PATH then becomes cars50_step_pool.json, and the finer scheme gets its own file rather than overwriting the 3-move one. Notebook 04 then reads prompts/cars50_step.txt, which is already there — a baseline naming the eleven codes, for you to improve on.

## What you just built, and what happens to it

This is the **pool** — everything usable in the corpus, with its natural label imbalance intact. It is **not** your gold set, and its labels are **not** your labels: they are the original corpus authors' judgment, and you have not yet agreed with them about anything.

What those labels are for is narrow, and worth being precise about:

1. **Stratifying the draw** in notebook 02 — you cannot sample evenly across classes without knowing what the classes are.
2. **A comparison** in notebook 03 — once you have annotated blind and adjudicated, `compare_to_published` shows you every item where your group landed somewhere different. That gap is evidence, and one of the more interesting things you can put in a report.

They are never the answer key you score the model against. That file does not exist yet — you make it in notebook 03.

---

**Next:** open `02_sample.ipynb`. It reads `POOL_PATH` — the file the cell above just wrote, in your group's Drive folder. Nothing to copy, nothing to paste: that path is the handoff, and both notebooks get it from the same `config.yaml`.